# Module 5 Guided Lab: Basic Data Transformation with Pandas

**BAN 6003: Data Management and Analytics Integration**

In this lab, we use the `nycflights13` dataset to practice the core transformation moves that analysts use all the time:

- filter rows
- select columns
- sort rows
- rename columns
- create derived columns
- export a transformed dataset

This module is intentionally focused. We are **not** doing aggregation, reshaping, joins, or heavy validation this week. Those topics come later.

## Module 5 Learning Goals

By the end of this guided lab, you should be able to:

1. Filter observations using `query()`.
2. Sort rows using `sort_values()`.
3. Select columns using `[[...]]`, `.loc[]`, and `.filter()`.
4. Rename variables using `.rename()`.
5. Create derived columns using `.assign()`.
6. Explain how a transformation changes the meaning or usability of a dataset.
7. Export a transformed dataset for later work.

The main idea is simple: **transformation changes raw columns into more useful analytical variables.**

## Part 0: Setup

We will use the Python version of the classic `nycflights13` dataset. It contains flight records from New York City airports in 2013.

If the package is not installed in your Codespace, the cell below will try to install it. In the course assignment, it is better if `nycflights13` is already included in `requirements.txt`.

In [ ]:
import sys
import subprocess

try:
    import nycflights13
except ImportError:
    print("nycflights13 is not installed. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nycflights13"])
    import nycflights13

import pandas as pd

print("Setup complete.")

## Part 1: Load the Flights Dataset

The main table we will use is `flights`.

Each row represents one scheduled flight from a New York City airport in 2013. This row-level meaning matters. When we filter, sort, select, or create variables, we are still working at the **flight level**.

In [ ]:
flights = nycflights13.flights.copy()

flights.head()

In [ ]:
flights.shape

In [ ]:
flights.columns

### Quick orientation

Before transforming anything, take a quick look at the variables. Some important columns for this lab are:

- `month`, `day`: date components
- `dep_time`, `arr_time`: actual departure and arrival times
- `dep_delay`, `arr_delay`: departure and arrival delays in minutes
- `carrier`: airline carrier code
- `origin`, `dest`: origin and destination airport codes
- `air_time`: flight time in minutes
- `distance`: distance in miles

We are not cleaning these variables this week. We are using them to practice transformation grammar.

In [ ]:
flights[["month", "day", "dep_delay", "arr_delay", "carrier", "origin", "dest", "air_time", "distance"]].head()

## Part 2: Filtering Rows with `query()`

Filtering means keeping only the rows that meet a condition.

In business analytics, filtering is how we narrow the data to a relevant scenario. For example:

- flights in January
- flights to Houston
- flights that arrived more than two hours late
- flights that left on time but arrived late

The `query()` method lets us write filtering conditions in a readable way.

In [ ]:
# Flights in January
january_flights = flights.query("month == 1")

january_flights.head()

In [ ]:
# Flights on February 2
feb_2_flights = flights.query("month == 2 & day == 2")

feb_2_flights.head()

In [ ]:
# Flights on February 2 that departed late
feb_2_late_departures = flights.query("month == 2 & day == 2 & dep_delay > 0")

feb_2_late_departures.head()

### Common comparison operators in `query()`

| Operator | Meaning | Example |
|---|---|---|
| `==` | equals | `month == 1` |
| `!=` | not equal | `month != 12` |
| `>` | greater than | `arr_delay > 120` |
| `>=` | greater than or equal to | `arr_delay >= 120` |
| `<` | less than | `dep_delay < 0` |
| `<=` | less than or equal to | `dep_delay <= 0` |
| `in` | belongs to a list | `dest in ['IAH', 'HOU']` |

Use parentheses in your own thinking, even when Pandas does not require them, because filtering logic can get confusing quickly.

In [ ]:
# Flights in November or December
holiday_season_flights = flights.query("month in [11, 12]")

holiday_season_flights.head()

In [ ]:
# Flights that arrived more than two hours late
very_late_arrivals = flights.query("arr_delay >= 120")

very_late_arrivals.head()

In [ ]:
# Flights with missing tail numbers
missing_tailnum = flights.query("tailnum.isna()", engine="python")

missing_tailnum.head()

### `query()` versus Boolean indexing

You may also see filtering written with Boolean indexing. This is equivalent to one of the examples above, but it is more verbose.

Both approaches are useful. In this course, `query()` is often easier to read for beginner-friendly filtering.

In [ ]:
feb_2_late_departures_alt = flights[
    (flights["month"] == 2) &
    (flights["day"] == 2) &
    (flights["dep_delay"] > 0)
]

feb_2_late_departures_alt.head()

### Your Turn 1: Filtering

Use `query()` to answer the following. For each task, show the number of matching rows using `.shape[0]`.

1. How many flights had an arrival delay of two or more hours?
2. How many flights flew to Houston, meaning destination is either `IAH` or `HOU`?
3. How many flights arrived more than two hours late but did not leave late?

In [ ]:
# 1. Flights with arrival delay of two or more hours
# Your code here

In [ ]:
# 2. Flights to Houston: IAH or HOU
# Your code here

In [ ]:
# 3. Flights that arrived more than two hours late but did not leave late
# Your code here

## Part 3: Sorting Rows with `sort_values()`

Sorting does not remove rows. It changes the order so that important cases rise to the top.

This is useful when you want to inspect extremes, such as:

- the most delayed flights
- the earliest departures
- the longest flights
- the shortest flights

Sorting is a transformation because it changes how we review and interpret the data.

In [ ]:
# Flights with the smallest departure delays first
flights.sort_values("dep_delay").head()

In [ ]:
# Flights with the largest departure delays first
flights.sort_values("dep_delay", ascending=False).head()

In [ ]:
# Sort by multiple variables
# First by departure delay, then by arrival delay
flights.sort_values(["dep_delay", "arr_delay"]).head()

In [ ]:
# Mixed sort directions:
# departure delay descending, arrival delay ascending
flights.sort_values(["dep_delay", "arr_delay"], ascending=[False, True]).head()

### Your Turn 2: Sorting

Use `sort_values()` to answer the following:

1. Which flights had the largest departure delays?
2. Which flights left earliest based on `dep_time`?
3. Which flights traveled the longest distance?
4. Which flights traveled the shortest distance?

Use `.head()` so the output is manageable.

In [ ]:
# 1. Largest departure delays
# Your code here

In [ ]:
# 2. Earliest departure times
# Your code here

In [ ]:
# 3. Longest distances
# Your code here

In [ ]:
# 4. Shortest distances
# Your code here

## Part 4: Selecting Columns

Selecting columns means keeping only the variables you need.

This is an important transformation habit because real datasets often have many columns. A smaller working DataFrame is easier to read, easier to debug, and easier to explain.

### Single brackets versus double brackets

A very common beginner issue is the difference between selecting one column as a Series and selecting one column as a DataFrame.

In [ ]:
one_column_series = flights["year"]

type(one_column_series)

In [ ]:
one_column_dataframe = flights[["year"]]

type(one_column_dataframe)

### Selecting multiple columns

To select multiple columns, use a list of column names inside double brackets.

In [ ]:
date_delay_columns = flights[["year", "month", "day", "dep_delay", "arr_delay"]]

date_delay_columns.head()

In [ ]:
# Equivalent selection using .loc[row_selection, column_selection]
date_delay_columns_loc = flights.loc[:, ["year", "month", "day", "dep_delay", "arr_delay"]]

date_delay_columns_loc.head()

### Finding columns by keyword

The `.filter(like=...)` method is useful when column names share a pattern.

For example, many variables related to departure include `"dep"` in the name.

In [ ]:
flights.filter(like="dep").head()

### Selecting from a requested list safely

Sometimes you have a list of desired columns, but not every name exists in the DataFrame. The `.columns.intersection()` approach keeps only names that actually exist.

This is useful when you are working with changing datasets or user-provided column lists.

In [ ]:
requested_columns = ["MONTH", "month", "day", "dep_delay", "arr_delay"]

available_columns = flights.columns.intersection(requested_columns)

available_columns

In [ ]:
flights[available_columns].head()

### Your Turn 3: Selecting Columns

1. What happens if you include the same variable more than once inside double brackets?
2. Select the columns included in this list, but only if they exist in the dataset:

```python
var_list = ["MONTH", "month", "day", "dep_delay", "arr_delay"]
```

3. Select all columns that contain `"time"` in the name.

In [ ]:
# 1. Include a variable multiple times
# Your code here

In [ ]:
# 2. Select only available variables from the list
var_list = ["MONTH", "month", "day", "dep_delay", "arr_delay"]

# Your code here

In [ ]:
# 3. Select variables that contain "time" in the name
# Your code here

## Part 5: Renaming Columns

Renaming is a small change that can make a dataset much easier to understand.

In practice, raw variable names are often abbreviated or inconsistent. We do not always rename every variable, but we often rename key variables to make later notebooks and reports clearer.

In [ ]:
flight_sml = flights[["year", "month", "day", "arr_delay", "dep_delay", "air_time", "distance"]].copy()

flight_sml.head()

In [ ]:
flight_sml = flight_sml.rename(columns={
    "arr_delay": "arrival_delay",
    "dep_delay": "departure_delay"
})

flight_sml.head()

### Naming tip

Use clear `snake_case` names for new columns:

- good: `arrival_delay`, `departure_delay`, `speed_mph`
- harder to read: `arrDelay`, `x1`, `newvar`, `delay2`

Good names help future you, your teammates, and your instructor understand your work.

## Part 6: Creating Derived Columns with `assign()`

Derived columns are new variables created from existing variables.

This is one of the most important transformation skills in analytics. Derived variables often become features for modeling, KPIs for reporting, or business indicators for decision-making.

For this lab, we will create:

- `gain`: arrival delay minus departure delay
- `hours`: air time converted from minutes to hours
- `speed_mph`: distance divided by hours

In [ ]:
flight_sml = flights[["year", "month", "day", "arr_delay", "dep_delay", "air_time", "distance"]].copy()

flight_sml = flight_sml.assign(
    gain = flight_sml["arr_delay"] - flight_sml["dep_delay"],
    hours = flight_sml["air_time"] / 60
)

flight_sml.head()

In [ ]:
flight_sml = flight_sml.assign(
    speed_mph = flight_sml["distance"] / flight_sml["hours"]
)

flight_sml.head()

### Optional: Chaining `assign()`

The code below creates multiple variables in one chain. This is compact, but it can be harder for beginners to read.

In your own work, clarity is more important than being clever.

In [ ]:
flight_sml_chained = (
    flights[["year", "month", "day", "arr_delay", "dep_delay", "air_time", "distance"]]
    .assign(
        gain = lambda x: x["arr_delay"] - x["dep_delay"],
        hours = lambda x: x["air_time"] / 60,
        speed_mph = lambda x: x["distance"] / (x["air_time"] / 60)
    )
)

flight_sml_chained.head()

### Your Turn 4: Creating Derived Columns

Create a smaller DataFrame called `flight_efficiency` that contains:

- `air_time`
- `distance`

Then create:

1. `distance_km`, converting distance from miles to kilometers using `1 mile = 1.61 kilometers`.
2. `minutes_per_km`, using `air_time / distance_km`.

Show the first few rows.

In [ ]:
# Create flight_efficiency with air_time and distance
# Your code here

In [ ]:
# Add distance_km
# Your code here

In [ ]:
# Add minutes_per_km
# Your code here

## Part 7: Putting the Basic Transformation Steps Together

Now we combine the main skills from this module:

1. filter rows
2. select columns
3. rename columns
4. create derived columns
5. sort rows
6. export the transformed dataset

This is not a final project pipeline yet. It is a small example of how transformation steps can be organized in a readable way.

In [ ]:
# Step 1: filter to flights with non-missing air_time and distance
# We are keeping this simple; more formal validation comes later.
transformed_flights = flights.query("air_time.notna() & distance.notna()", engine="python")

# Step 2: select useful columns
transformed_flights = transformed_flights[[
    "year", "month", "day", "carrier", "origin", "dest",
    "dep_delay", "arr_delay", "air_time", "distance"
]]

# Step 3: rename columns for clarity
transformed_flights = transformed_flights.rename(columns={
    "dep_delay": "departure_delay",
    "arr_delay": "arrival_delay"
})

# Step 4: create derived columns
transformed_flights = transformed_flights.assign(
    delay_gain = transformed_flights["arrival_delay"] - transformed_flights["departure_delay"],
    air_time_hours = transformed_flights["air_time"] / 60,
    speed_mph = transformed_flights["distance"] / (transformed_flights["air_time"] / 60)
)

# Step 5: sort to inspect the highest speed values
transformed_flights = transformed_flights.sort_values("speed_mph", ascending=False)

transformed_flights.head()

### Short reflection

Write 2–3 sentences below.

1. What new variables did we create?
2. Why might these variables be useful for analysis?
3. What would you want to check later before trusting these values?

This is not a full transformation summary yet. Formal documentation and data dictionaries come later. For now, practice explaining what your transformations mean.

**Your response here:**

- 
- 
- 

## Part 8: Export the Transformed Dataset

Exporting gives us a saved output that can be reused later.

For now, we will export a CSV file. Later in the course, we will discuss other formats and database storage.

In [ ]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "flights_transformed_module5.csv"

transformed_flights.to_csv(output_path, index=False)

print(f"Saved transformed dataset to: {output_path}")

## Lab Wrap-Up

In this lab, you practiced the core Pandas transformation moves:

- `query()` for filtering rows
- `sort_values()` for ordering rows
- `[[...]]`, `.loc[]`, and `.filter()` for selecting columns
- `rename()` for clearer variable names
- `assign()` for derived columns
- `to_csv()` for exporting a transformed dataset

Next module, we will focus on **string and categorical data transformation**, using a small messy dataset designed to show why text values and categories often need special attention.